In [5]:
import cv2
import numpy as np
import datetime
import csv
import json
import base64
from ultralytics import YOLO
from io import BytesIO
from PIL import Image

# Load the pre-trained YOLO model
model = YOLO("models/yolov8m.pt")



def process_image_frame(image, frame_number, video_start_time, total_frames, fps, visual_debugging=False):
    # Define the required polygon, lines, and other parameters
    counter_polygon_points = [(5, 520), (5, 690), (530, 690), (530, 520), (5, 520)]  # Counter
    exit_line = [(825, 320), (1020, 320)]  # Exit line LEFT
    entrance_line = [(1270, 325), (1435, 325)]  # Entrance line RIGHT
    exit_polygon_points = [(825, 300), (825, 370), (1020, 370), (1025, 300), (825, 300)]  # Exit
    entrance_polygon_points = [(1270, 300), (1270, 370), (1435, 370), (1435, 300), (1320, 300)]  # Entrance
    classes_to_count = [0]  # Class ID for persons

    # Memory system for tracking counts across frames
    counted_objects = {
        "exit": set(),     # Set of object IDs that have been counted for exiting
        "entrance": set()  # Set of object IDs that have been counted for entering
    }

    # Initialize counters
    polygon_count = 0

    # Perform object tracking on the current frame
    results = model.track(image, persist=True, show=False, classes=classes_to_count)[0]

    # Draw the counter polygon and lines only if visual_debugging is True
    if visual_debugging:
        cv2.polylines(image, [np.array(counter_polygon_points, np.int32)], isClosed=True, color=(0, 255, 0), thickness=2)
        cv2.line(image, exit_line[0], exit_line[1], (0, 165, 255), 2)  # Exit line
        cv2.line(image, entrance_line[0], entrance_line[1], (0, 165, 255), 2)  # Entrance line
        cv2.polylines(image, [np.array(exit_polygon_points, np.int32)], isClosed=True, color=(0, 165, 255), thickness=2)
        cv2.polylines(image, [np.array(entrance_polygon_points, np.int32)], isClosed=True, color=(0, 165, 255), thickness=2)

    # Process each tracked object
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Extract bounding box coordinates
        class_id = int(box.cls[0])
        confidence = box.conf[0]
        track_id = int(box.id[0])  # Object ID from tracking

        # Check if the detected class is one we want to count
        if class_id in classes_to_count:
            # Adjust the bounding box to be smaller and positioned at the feet
            box_height = y2 - y1
            new_y2 = y2 + box_height // 5  # Move down by 1/5th of the height for foot positioning
            new_y1 = new_y2 - box_height // 5 * 2  # Set the new height to be smaller (2/5ths of original)
            adjusted_box = (x1, new_y1, x2, new_y2)

            if visual_debugging:
                cv2.rectangle(image, (adjusted_box[0], adjusted_box[1]), (adjusted_box[2], adjusted_box[3]), (255, 0, 0), 2)
                label = f"ID: {track_id}, Conf: {confidence:.2f}"
                cv2.putText(image, label, (adjusted_box[0], adjusted_box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)


            # Check if the object's top-left point is inside the counter polygon
            if point_in_polygon((adjusted_box[0], adjusted_box[1]), counter_polygon_points):
                polygon_count += 1

            # Check if the object's top-left point is inside the exit polygon
            if point_in_polygon((adjusted_box[0], adjusted_box[1]), exit_polygon_points):
                if crosses_line((adjusted_box[0], adjusted_box[1]), exit_line[0], exit_line[1], "up") and track_id not in counted_objects["exit"]:
                    counted_objects["exit"].add(track_id)

            # Check if the object's top-left point is inside the entrance polygon
            if point_in_polygon((adjusted_box[0], adjusted_box[1]), entrance_polygon_points):
                if crosses_line((adjusted_box[0], adjusted_box[1]), entrance_line[0], entrance_line[1], "down") and track_id not in counted_objects["entrance"]:
                    counted_objects["entrance"].add(track_id)

    # Create frame metadata
    metadata = {
        "frame_number": frame_number,
        "total_frames": total_frames,
        "exit_count": len(counted_objects["exit"]),
        "entrance_count": len(counted_objects["entrance"]),
        "counter": polygon_count
    }

    # Display counts for the counter polygon, exit, and entrance
    cv2.putText(image, f"Counter Count: {polygon_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(image, f"Exit Count: {len(counted_objects['exit'])}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(image, f"Entrance Count: {len(counted_objects['entrance'])}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)


    # Convert image to base64
    _, buffer = cv2.imencode('.jpg', image)
    frame_base64 = base64.b64encode(buffer).decode('utf-8')

    # Yield the frame and metadata in JSON format
    yield json.dumps({
        "frame": frame_base64,
        "metadata": metadata
    })

# Helper functions used in process_frame
def point_in_polygon(point, polygon):
    return cv2.pointPolygonTest(np.array(polygon, np.int32), point, False) >= 0

def crosses_line(top_left_point, line_start, line_end, direction):
    if direction == "up":
        return top_left_point[1] < line_start[1]
    elif direction == "down":
        return top_left_point[1] > line_start[1]
    return False


In [6]:
import cv2
import datetime
import json
# Open the video file
video_name = "ch03_20240522000000.mp4"
cap = cv2.VideoCapture(video_name)

# Check if the video was successfully opened
assert cap.isOpened(), "Error reading video file"

# Get video properties: total frame count and frames per second (fps)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Function to get the video start time from the filename
def get_video_starttime(video_name):
    video_start_time = video_name.split("_")[1].strip(".mp4")
    return datetime.datetime.strptime(video_start_time, "%Y%m%d%H%M%S")

# Get the video start time
video_start_time = get_video_starttime(video_name)

# Initialize an empty list to store the JSON results for each frame
results_list = []

# Set visual debugging option
visual_debugging = True  # Change to False for plain frames

# Process each frame
for frame_number in range(total_frames):
    success, frame = cap.read()
    if not success:
        print("End of video or error reading frame.")
        break

    # Call the generator function with the frame
    result_generator = process_image_frame(frame, frame_number, video_start_time, total_frames, fps, visual_debugging)

    # Get the result from the generator
    result_json_str = next(result_generator)

    # Parse the JSON string to a dictionary
    result_json = json.loads(result_json_str)

    # Append the result to the results list
    results_list.append(result_json)

    # Optionally, print a message indicating progress
    print(f"Processed frame {frame_number + 1}/{total_frames}")

    # Decode the base64 image from the JSON result
    frame_base64 = result_json["frame"]
    nparr = np.frombuffer(base64.b64decode(frame_base64), np.uint8)
    decoded_frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Display the frame
    if visual_debugging:
        cv2.imshow("Processed Frame", decoded_frame)

    # Wait for a key press; if 'q' is pressed, exit the loop
    if cv2.waitKey(int(1000 / fps)) & 0xFF == ord('q'):
        break

# Release the video capture object and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()

# Combine all the frame results into a single JSON object
final_result = {
    "video_name": video_name,
    "total_frames": total_frames,
    "fps": fps,
    "results": results_list
}

# Convert the combined result into JSON format
final_result_json = json.dumps(final_result, indent=4)

# Save the JSON to a file (optional)
with open("test.json", "w") as json_file:
    json_file.write(final_result_json)

# Print the final result (or you can just inspect the file)
print("Processing complete. JSON result saved to 'test.json'.")



0: 384x640 11 persons, 546.8ms
Speed: 8.4ms preprocess, 546.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 1/277

0: 384x640 11 persons, 453.9ms
Speed: 5.1ms preprocess, 453.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 2/277

0: 384x640 12 persons, 511.2ms
Speed: 3.3ms preprocess, 511.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 3/277

0: 384x640 12 persons, 489.0ms
Speed: 11.6ms preprocess, 489.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 4/277

0: 384x640 12 persons, 611.3ms
Speed: 13.6ms preprocess, 611.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 5/277

0: 384x640 12 persons, 549.2ms
Speed: 18.1ms preprocess, 549.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 6/277

0: 384x640 12 persons, 469.6ms
Speed: 5.4ms preprocess, 469.6ms inference, 2.8ms pos